# Quickstart Tutorial
This tutorial walks through the end-to-end `llm_perf` workflow: configuring the environment, loading specs (model/system/partition/tuner), optionally converting a HuggingFace config, and running the `InferenceCalculator` to inspect latency, memory, and communication diagnostics.

## 1. Environment Setup and Imports
We verify the local environment, import the public `llm_perf` APIs, and create small helper utilities for nicely displaying dataclasses and tabular summaries.

In [8]:
from __future__ import annotations

import json
import os
import platform
import shutil
import subprocess
from dataclasses import asdict
from pathlib import Path
from typing import Any, Dict

from IPython.display import JSON, display

from llm_perf import InferenceCalculator
from llm_perf.io import (
    list_hw_system_ids,
    list_model_ids,
    list_partition_ids,
    list_tuner_ids,
    load_model_from_db,
    load_model_spec,
    load_partition_from_db,
    load_system_from_db,
    load_tuner_from_db,
)


REPO_ROOT = Path.cwd()
DB_ROOT = REPO_ROOT / "llm_perf" / "database"

MODEL_ID = "example.model.dense"
SYSTEM_ID = "example.sys"
PARTITION_ID = "example.partition"
TUNER_ID = "example.tuner"

print(f"Python {platform.python_version()} running on {platform.platform()}")
print(f"Repository root: {REPO_ROOT}")

print("\nAvailable Database IDs\n=======================")
print("Models:", list_model_ids())
print("Systems:", list_hw_system_ids())
print("Partitions:", list_partition_ids())
print("Tuners:", list_tuner_ids())

Python 3.12.10 running on Windows-11-10.0.26200-SP0
Repository root: c:\Users\lvyue\OneDrive\Projects\Data Center Interconnect\llm.partition.materials\python\llm_perf

Available Database IDs
Models: ['deepseekv31', 'deepseekv31_terminus', 'example.model.dense', 'example.model.moe', 'qwen3_vl_235b_fp8']
Systems: ['example.sys', 'h100.32gpu', 'imaginary.32npu']
Partitions: ['example.partition']
Tuners: ['example.tuner', 'qwen3.tuner']


## 2. Load and Inspect Model Card (Optional HF Fetch)
We load a ready-to-use model spec from the on-disk database, compute a few derived stats (head dimension, bytes per parameter, estimated parameter footprint), and optionally show how to adapt a HuggingFace config into the same schema.

In [9]:
model_spec = load_model_from_db(MODEL_ID)
print("\nModel Spec\n==========")
display(JSON(asdict(model_spec), expanded=False))


Model Spec


<IPython.core.display.JSON object>

**Optional: Convert a HuggingFace config into an `llm_perf` model schema.**
Run the next cell if you have a HF `config.json` available (one sample is already staged under `llm_perf/database/model/external.model/hf/`).

In [10]:
from llm_perf.utils import (
    convert_hf_config_to_model_json,
)

HF_SAMPLE = DB_ROOT / "model" / "external.model" / "hf" / "qwen3_vl_235b_a22b_thinking_fp8.json"
HF_ADAPTED_OUTPUT = DB_ROOT / "model" / "qwen3_vl_235b_fp8.json"

converted_spec = None
if HF_SAMPLE.exists():
    print(f"Adapting HuggingFace config → llm_perf JSON: {HF_SAMPLE}")
    converted_path = convert_hf_config_to_model_json(
        hf_config_path=HF_SAMPLE,
        out_path=HF_ADAPTED_OUTPUT,
        name_override="qwen3_vl_235b_fp8",
        overwrite=True,
    )
    print(f"Wrote adapted model card to {converted_path}")
    converted_spec = load_model_spec(converted_path)
    display(JSON(asdict(converted_spec), expanded=False))
else:
    print("No HF config found at", HF_SAMPLE)

print("You can now treat the adapted file like any other entry in the llm_perf database.")

if converted_spec is not None:
    MODEL_ID = converted_spec.name
    model_spec = load_model_from_db(MODEL_ID)
    print("\nConverted Model Spec\n====================")
    display(JSON(asdict(model_spec), expanded=False))
else:
    print("Skipping converted spec reload because no HF config was adapted.")

Adapting HuggingFace config → llm_perf JSON: c:\Users\lvyue\OneDrive\Projects\Data Center Interconnect\llm.partition.materials\python\llm_perf\llm_perf\database\model\external.model\hf\qwen3_vl_235b_a22b_thinking_fp8.json
Wrote adapted model card to c:\Users\lvyue\OneDrive\Projects\Data Center Interconnect\llm.partition.materials\python\llm_perf\llm_perf\database\model\qwen3_vl_235b_fp8.json


<IPython.core.display.JSON object>

You can now treat the adapted file like any other entry in the llm_perf database.

Converted Model Spec


<IPython.core.display.JSON object>

## 3. Collect and Display Hardware System Info
Next we inspect the hardware/network description used by the simulator by loading the baked JSON spec from the database.

In [11]:
system_spec = load_system_from_db(SYSTEM_ID)
print("\nHardware / Network Spec (from database)\n========================================")
display(JSON(asdict(system_spec), expanded=False))


Hardware / Network Spec (from database)


<IPython.core.display.JSON object>

## 4. Define and Visualize Partition Strategy
`PartitionSpec` controls data/pipeline/tensor/expert/sequence sharding. We inspect the JSON-backed plan and plot the scale of each axis to spot imbalances quickly.

In [12]:
partition_spec = load_partition_from_db(PARTITION_ID)
print("\nPartition Spec\n==============")
display(JSON(asdict(partition_spec), expanded=False))

dims = {
    "DP": partition_spec.DP,
    "PP": partition_spec.PP,
    "EP": partition_spec.EP,
    "TP": partition_spec.TP,
    "SP": partition_spec.SP,
}

print("\nParallelism Factors\n====================")
for axis, value in dims.items():
    bar = "#" * min(value, 40)
    print(f"{axis}: {value:>4} | {bar}")

print("\nDevice Budget Cross-Check\n==========================")
total_devices = (
    partition_spec.DP
    * partition_spec.PP
    * max(1, partition_spec.EP)
    * partition_spec.TP
    * max(1, partition_spec.SP)
)
print(
    "Nested replicas = DP({dp}) → PP({pp}) → EP({ep}) → TP({tp}) → SP({sp}) = {total} devices".format(
        dp=partition_spec.DP,
        pp=partition_spec.PP,
        ep=partition_spec.EP,
        tp=partition_spec.TP,
        sp=partition_spec.SP,
        total=total_devices,
    )
)
print("Ensure this nested count does not exceed `system_spec.num_devices`.")


Partition Spec


<IPython.core.display.JSON object>


Parallelism Factors
DP:    1 | #
PP:    1 | #
EP:    8 | ########
TP:    4 | ####
SP:    2 | ##

Device Budget Cross-Check
Nested replicas = DP(1) → PP(1) → EP(8) → TP(4) → SP(2) = 64 devices
Ensure this nested count does not exceed `system_spec.num_devices`.


## 5. Configure and Display Tuner Parameters
Tuning knobs capture algorithm choices, overlap heuristics, and search-related metadata. We load them, surface the raw values, and log a few derived quantities for reproducibility.

In [13]:
tuner_spec = load_tuner_from_db(TUNER_ID)
print("\nTuning Spec\n===========")
display(JSON(asdict(tuner_spec), expanded=False))

derived_tuner = {
    "S_decode_tokens": tuner_spec.S_decode,
    "collectives_per_layer": {
        "TP": tuner_spec.n_TP_collectives,
        "EP": tuner_spec.n_EP_collectives,
        "SP": tuner_spec.n_SP_collectives,
    },
    "overlap_factor": tuner_spec.overlap_factor,
    "tp_algorithm": tuner_spec.tp_algorithm,
    "ep_algorithm": tuner_spec.ep_algorithm,
}

print("\nDerived tuner notes\n====================")
display(JSON(derived_tuner, expanded=False))


Tuning Spec


<IPython.core.display.JSON object>


Derived tuner notes


<IPython.core.display.JSON object>

## 6. Execute Inference and Surface Diagnostics
With all specs in hand we can instantiate `InferenceCalculator`, run the analytical models, and report memory, FLOPs, traffic, communication, and latency breakdowns similar to `test_example.py`. These metrics let you validate hardware fit and throughput before spinning up a real cluster.

In [14]:
calculator = InferenceCalculator(model_spec, system_spec, partition_spec, tuner_spec)
results = calculator.run()

print("\nMemory (per device)\n====================")
print(f"Params: {results.memory.M_theta_device / 1e9:.3f} GB")
print(f"Activations: {results.memory.M_act_device / 1e9:.3f} GB")
print(f"KV cache: {results.memory.M_kv_device / 1e9:.3f} GB")
print(f"Total: {results.memory.M_total_device / 1e9:.3f} GB (fits HBM? {results.memory.fits_in_HBM})")

print("\nFLOPs\n=====")
print(f"Per token per device: {results.flops.F_token_device / 1e9:.3f} GFLOPs")

print("\nTraffic\n=======")
print(f"Per token traffic: {results.traffic.T_token_eff / 1e9:.3f} GB")

print("\nCommunication\n=============")
print(f"TP time: {results.comm.t_TP * 1e6:.3f} us")
print(f"EP time: {results.comm.t_EP * 1e6:.3f} us")
print(f"SP time: {results.comm.t_SP * 1e6:.3f} us")
print(f"PP time: {results.comm.t_PP * 1e6:.3f} us")
print(f"Stage comm: {results.comm.t_comm_stage * 1e6:.3f} us")

print("\nLatency / Throughput\n=====================")
print(f"t_compute: {results.latency.t_compute * 1e6:.3f} us")
print(f"t_mem: {results.latency.t_mem * 1e6:.3f} us")
print(f"t_local: {results.latency.t_local * 1e6:.3f} us")
print(f"t_comm: {results.latency.t_comm * 1e6:.3f} us")
print(f"t_token: {results.latency.t_token * 1e6:.3f} us")
print(f"Single replica TPS: {results.latency.TPS_single:.3f}")
print(f"Total TPS (DP-scaled): {results.latency.TTPS:.3f}")

summary_bundle = {
    "memory": asdict(results.memory),
    "flops": asdict(results.flops),
    "traffic": asdict(results.traffic),
    "comm": asdict(results.comm),
    "latency": asdict(results.latency),
}
print("\nStructured results (JSON)\n=========================")
display(JSON(summary_bundle, expanded=False))


Memory (per device)
Params: 7.721 GB
Activations: 0.002 GB
KV cache: 0.012 GB
Total: 7.734 GB (fits HBM? True)

FLOPs
=====
Per token per device: 1.651 GFLOPs

Traffic
Per token traffic: 7.579 GB

Communication
TP time: 6.015 us
EP time: 14.143 us
SP time: 1.328 us
PP time: 5.020 us
Stage comm: 2590.186 us

Latency / Throughput
t_compute: 1.651 us
t_mem: 2262.447 us
t_local: 2262.447 us
t_comm: 2590.186 us
t_token: 4852.633 us
Single replica TPS: 206.074
Total TPS (DP-scaled): 206.074

Structured results (JSON)


<IPython.core.display.JSON object>